# Gerador de Relatório Mensal (por Ano/Órgão)

Este notebook carrega o arquivo mestre de um ano/órgão, agrupa os totais por mês (garantindo 12 meses, preenchendo com 0 se ausente) e salva o relatório CSV em uma subpasta (`.../Relatorios_Mensais/`).

## 1. Importações e Configurações

In [ ]:
import pandas as pd
import numpy as np
import os

# --- Configurações ---
ano = 2025 # Ano dos dados a serem processados e salvos
orgao = 'UFPB'
# orgao = 'UFCG'

# --- Caminho para o ARQUIVO DE ENTRADA (Master) ---
pasta_dados_ano = f'dadosViagens/dados_viagens{ano}'
arquivo_entrada = os.path.join(pasta_dados_ano, f'df_master_{orgao}_aereo_{ano}.csv') 

# --- Caminho para a PASTA DE SAÍDA (Relatórios Mensais) ---
PASTA_RELATORIOS_MENSAIS = os.path.join(pasta_dados_ano, 'Relatorios_Mensais')
os.makedirs(PASTA_RELATORIOS_MENSAIS, exist_ok=True)


# Colunas que serão usadas do arquivo de entrada
COL_ID_VIAGEM = 'Identificador do processo de viagem'
COL_DATA = 'Período - Data de início' 
COL_DISTANCIA = 'Distância (GCD)'
COL_EMISSOES = 'Emissões (KgCO2eq)'
COL_GASTOS_PASSAGENS = 'Valor passagens' 

print(f"CONFIGURAÇÃO: Processando {orgao} / {ano}")
print(f"-> Lendo de: {arquivo_entrada}")
print(f"-> Salvando em: {PASTA_RELATORIOS_MENSAIS}")

CONFIGURAÇÃO: Processando UFPB / 2023
-> Lendo de: dadosViagens/dados_viagens2023\df_master_UFPB_aereo_2023.csv
-> Salvando em: dadosViagens/dados_viagens2023\Relatorios_Mensais


## 2. Carregar e Preparar Dados

In [18]:
print(f"🔄 Carregando dados de: '{arquivo_entrada}'...")
df = pd.DataFrame() # Inicializa df vazio
try:
    df = pd.read_csv(arquivo_entrada)
    print(f"✅ Dados carregados com sucesso ({len(df)} viagens).")
    
    colunas_necessarias = [COL_ID_VIAGEM, COL_DATA, COL_DISTANCIA, COL_EMISSOES, COL_GASTOS_PASSAGENS]
    colunas_faltantes = [col for col in colunas_necessarias if col not in df.columns]
    if colunas_faltantes:
        print(f"❌ ERRO: Colunas essenciais não encontradas: {colunas_faltantes}")
        df = pd.DataFrame() # Reseta df se faltar algo crítico
    else:
        print("✅ Colunas necessárias (Data, Distância, Emissões, Passagens) encontradas.")

except FileNotFoundError:
    print(f"❌ ERRO: Arquivo '{arquivo_entrada}' não encontrado.")
except Exception as e:
    print(f"❌ Ocorreu um erro ao carregar os dados: {e}")

# --- Processamento de Datas e Custos --- 
if not df.empty:
    print("🔄 Processando datas e custos...")
    df['Data_Viagem'] = pd.to_datetime(df[COL_DATA], format='%d/%m/%Y', errors='coerce')
    
    colunas_numericas_processar = [COL_DISTANCIA, COL_EMISSOES, COL_GASTOS_PASSAGENS]
    for col in colunas_numericas_processar:
         df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.', regex=False), errors='coerce').fillna(0)
    print("✅ Colunas de Distância, Emissões e Passagens convertidas para numérico.")
    
    df.dropna(subset=['Data_Viagem'], inplace=True)
    
    df['Mes_Num'] = df['Data_Viagem'].dt.month 
    df['Mes_Str'] = df['Data_Viagem'].dt.strftime('%m') 
    df['Mes_Ano'] = f"{ano}-" + df['Mes_Str'] 
    
    print("✅ Datas processadas e colunas de Mês criadas (ANO FORÇADO PELO CONFIG).")
else:
    print("⚠️ DataFrame vazio, processamento pulado.")

🔄 Carregando dados de: 'dadosViagens/dados_viagens2023\df_master_UFPB_aereo_2023.csv'...
✅ Dados carregados com sucesso (242 viagens).
✅ Colunas necessárias (Data, Distância, Emissões, Passagens) encontradas.
🔄 Processando datas e custos...
✅ Colunas de Distância, Emissões e Passagens convertidas para numérico.
✅ Datas processadas e colunas de Mês criadas (ANO FORÇADO PELO CONFIG).


## 3. Agrupar por Mês e Salvar Relatório

In [19]:
df_mensal = pd.DataFrame() # Cria DF vazio por segurança

if not df.empty:
    print(f"🔄 Agrupando totais por Mês/Ano para {ano}...")
    
    # 1. Agrupa os dados que existem
    df_agrupado = df.groupby(['Mes_Ano', 'Mes_Num']).agg(
        Total_Distancia_Km = (COL_DISTANCIA, 'sum'),
        Total_Emissoes_KgCO2eq = (COL_EMISSOES, 'sum'),
        Total_Viagens = (COL_ID_VIAGEM, 'count'),
        Total_Passagens = (COL_GASTOS_PASSAGENS, 'sum') 
    ).reset_index()

    # 2. Cria o template de 12 meses para o ano configurado
    meses_template = pd.DataFrame({'Mes_Num': range(1, 13)})
    meses_template['Mes_Ano'] = meses_template['Mes_Num'].apply(lambda x: f"{ano}-{x:02d}")

    # 3. Mescla o template com os dados agrupados
    df_mensal = pd.merge(
        meses_template, 
        df_agrupado, 
        on=['Mes_Num', 'Mes_Ano'], 
        how='left' # Mantém todos os 12 meses do template
    )

    # 4. Preenche os meses sem viagens com 0
    colunas_para_zerar = [
        'Total_Distancia_Km', 'Total_Emissoes_KgCO2eq', 
        'Total_Viagens', 'Total_Passagens'
    ]
    df_mensal[colunas_para_zerar] = df_mensal[colunas_para_zerar].fillna(0)
    
    # 5. Garante os tipos corretos (especialmente para 'Total_Viagens')
    df_mensal['Total_Viagens'] = df_mensal['Total_Viagens'].astype(int)
    df_mensal['Mes_Num'] = df_mensal['Mes_Num'].astype(int)
    
    df_mensal.sort_values('Mes_Num', inplace=True)
    print("✅ Agrupamento concluído (garantido 12 meses).")
    
    print(f"\n--- Relatório Mensal: Totais de Viagens Aéreas ({orgao} {ano}) ---")
    colunas_exibir = ['Mes_Ano', 'Total_Viagens', 'Total_Distancia_Km', 'Total_Emissoes_KgCO2eq', "Total_Passagens"]
    print(df_mensal[colunas_exibir].to_string(index=False, formatters={
        'Total_Distancia_Km': '{:,.0f} km'.format,
        'Total_Emissoes_KgCO2eq': '{:,.0f} kg'.format,
        'Total_Passagens': 'R$ {:,.2f}'.format 
    }))
    
    try:
        nome_arquivo_saida = f"relatorio_mensal_{orgao}_aereo_{ano}.csv"
        caminho_saida = os.path.join(PASTA_RELATORIOS_MENSAIS, nome_arquivo_saida)
        df_mensal.round(2).to_csv(caminho_saida, index=False)
        print(f"\n✅ Relatório mensal (com 12 meses) salvo com sucesso em: '{caminho_saida}'")
    except Exception as e_save:
        print(f"❌ Erro ao salvar relatório mensal: {e_save}")
        
else:
    print("⚠️ DataFrame vazio, agrupamento pulado.")

🔄 Agrupando totais por Mês/Ano para 2023...
✅ Agrupamento concluído (garantido 12 meses).

--- Relatório Mensal: Totais de Viagens Aéreas (UFPB 2023) ---
Mes_Ano  Total_Viagens Total_Distancia_Km Total_Emissoes_KgCO2eq Total_Passagens
2023-01              2           5,670 km               1,037 kg     R$ 5,816.67
2023-02              6          20,805 km               3,910 kg    R$ 19,179.99
2023-03             19          71,648 km              13,641 kg    R$ 64,000.22
2023-04             24          86,900 km              16,585 kg    R$ 96,906.21
2023-05             37         160,344 km              30,892 kg   R$ 139,615.77
2023-06             20          79,436 km              15,348 kg    R$ 48,892.66
2023-07             20          72,509 km              14,041 kg    R$ 55,479.42
2023-08             37         121,719 km              23,552 kg   R$ 100,979.98
2023-09             55         185,822 km              35,737 kg   R$ 147,870.32
2023-10             21          79,0